In [4]:
import pandas as pd

df = pd.read_csv("all-data.csv", encoding="latin-1", names=["sentiment", "text"])
print(df.shape)
print(df["sentiment"].value_counts(normalize=True))
print(df.sample(5, random_state=1))

(4846, 2)
sentiment
neutral     0.594098
positive    0.281263
negative    0.124639
Name: proportion, dtype: float64
     sentiment                                               text
910   positive  The pilot project proved that RIFD technology ...
1715  positive  Aspocomp has repaid its interest bearing liabi...
4700  negative  Operating profit fell to EUR 15.1 mn from EUR ...
2776   neutral  After the restructuring , UPM 's average paper...
4284  negative         This is bad news for the barbeque season .


In [5]:
from textblob import TextBlob

def lexicon_label(text, thresh=0.05):
    p = TextBlob(text).sentiment.polarity
    return "positive" if p > thresh else "negative" if p < -thresh else "neutral"

df["lex_pred"] = df["text"].apply(lexicon_label)
print((df["lex_pred"] == df["sentiment"]).mean())

0.5092860090796533


In [6]:
wrong = df[df["lex_pred"] != df["sentiment"]]
print(wrong[["text", "sentiment", "lex_pred"]].sample(8, random_state=1).to_string())

                                                                                                                                                                                                                                    text sentiment  lex_pred
709                                    Due to rapid expansion , the market share of Tallink in terms of passenger carriage through Tallinna Sadam Port of Tallinn totaled 55 percent in November against 45.7 percent in November 2005 .  positive  negative
1204                                                                                                                                          Currently more than 100 of Global Fortune 500 companies are using SSH security solutions .   neutral  positive
1995                                                                                                                                                              Excluding non-recurring items , pre-tax profit surged 45 % to EUR80m .  positiv

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)
print(len(X_train), len(X_test))


3876 970


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english",
                              ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)

pred = model.predict(X_test)
print((pred == y_test).mean())

0.7216494845360825


In [9]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

    negative       0.56      0.69      0.61       121
     neutral       0.81      0.79      0.80       576
    positive       0.62      0.59      0.60       273

    accuracy                           0.72       970
   macro avg       0.66      0.69      0.67       970
weighted avg       0.73      0.72      0.72       970



In [10]:
import numpy as np

vec, clf = model["tfidf"], model["clf"]
names = np.array(vec.get_feature_names_out())
for i, cls in enumerate(clf.classes_):
    print(cls, names[np.argsort(clf.coef_[i])[-10:]])

negative ['lay' 'dropped' 'negative' 'result' 'half' 'fall' 'staff' 'lower' 'fell'
 'decreased']
neutral ['range' 'include' 'stake' 'development' 'deal' 'disclosed' 'business'
 'includes' 'value' 'approximately']
positive ['positive' 'improved' 'agreement' 'leading' 'awarded' 'grew' 'signed'
 'rose' 'increased' 'increase']


In [11]:
import pandas as pd

errors = pd.DataFrame({"text": X_test, "true": y_test, "pred": pred})
errors = errors[errors["true"] != errors["pred"]]
print(errors.sample(10, random_state=1).to_string())

                                                                                                                                                                                                           text      true      pred
2805  Apartments of YIT Home may be purchased in 5 regions of Russia , where YIT subsidiaries carry out their activities : Moscow and Moscow region , St. Petersburg , Ekaterinburg , Kazan and Rostov-on-Don .   neutral  positive
731                                                                                                 RISING costs have forced packaging producer Huhtamaki to axe 90 jobs at its Hampshire manufacturing plant .  negative   neutral
1813                                                                                                        Swiss franc surged to record 1.42 euros after it reported sharpest gain in manufacturing in March .  positive   neutral
2175                                                                                    

In [12]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

model2 = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 3), min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(class_weight="balanced")),
])
model2.fit(X_train, y_train)
pred2 = model2.predict(X_test)
print((pred2 == y_test).mean())
print(classification_report(y_test, pred2))

0.7556701030927835
              precision    recall  f1-score   support

    negative       0.67      0.64      0.66       121
     neutral       0.81      0.84      0.82       576
    positive       0.67      0.63      0.65       273

    accuracy                           0.76       970
   macro avg       0.72      0.71      0.71       970
weighted avg       0.75      0.76      0.75       970



In [13]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, truncation=True)

sample = X_test.iloc[:5].tolist()
for s, out in zip(sample, finbert(sample)):
    best = max(out, key=lambda d: d["score"])
    print(best["label"], round(best["score"], 2), "|", s[:80])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

neutral 0.86 | Following the payment made in April , the company has a total of EUR 23.0 millio
neutral 0.95 | The share subscription period for C options will commence on 1 September 2008 an
neutral 0.68 | Aspocomp intends to set up a plant to manufacture printed circuit boards with an
positive 0.94 | Finnish Rautaruukki has been awarded a contract to supply and install steel supe
negative 0.98 | Finnish Bank of +_land reports its operating profit fell to EUR 4.9 mn in the th


In [14]:
import transformers, torch
print(transformers.__version__, torch.__version__)

5.22.1 2.15.0+cpu


In [15]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, truncation=True)

sample = X_test.iloc[:5].tolist()
for s, out in zip(sample, finbert(sample)):
    best = max(out, key=lambda d: d["score"])
    print(best["label"], round(best["score"], 2), "|", s[:80])

print(y_test.iloc[:5].tolist())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

neutral 0.86 | Following the payment made in April , the company has a total of EUR 23.0 millio
neutral 0.95 | The share subscription period for C options will commence on 1 September 2008 an
neutral 0.68 | Aspocomp intends to set up a plant to manufacture printed circuit boards with an
positive 0.94 | Finnish Rautaruukki has been awarded a contract to supply and install steel supe
negative 0.98 | Finnish Bank of +_land reports its operating profit fell to EUR 4.9 mn in the th
['neutral', 'neutral', 'neutral', 'positive', 'negative']


In [17]:
from sklearn.metrics import classification_report

pred_fb = [max(o, key=lambda d: d["score"])["label"].lower() for o in outs]

print((pd.Series(pred_fb, index=y_test.index) == y_test).mean())
print(classification_report(y_test, pred_fb))

0.8721649484536083
              precision    recall  f1-score   support

    negative       0.76      0.97      0.85       121
     neutral       0.95      0.84      0.89       576
    positive       0.79      0.90      0.84       273

    accuracy                           0.87       970
   macro avg       0.84      0.90      0.86       970
weighted avg       0.88      0.87      0.87       970



In [18]:
import torch
print(torch.cuda.is_available())


False
